# TP01 — Du SI à une première prédiction

**Introduction au Machine Learning — M2 MIAGE**  
Université Paris Dauphine – PSL

## Mission

Vous travaillez dans l'équipe Data d'une marketplace. Au moment où une commande vient d'être validée et où une date estimée de livraison a été communiquée au client, l'entreprise souhaite mieux anticiper son délai réel de livraison.

**Question métier : peut-on prédire le nombre de jours nécessaires pour livrer une commande ?**

Dans ce TP, l'objectif n'est pas seulement d'entraîner un modèle. Nous allons suivre toute la chaîne de raisonnement :

**SI → unité statistique → table analytique → variables disponibles → baseline → modèle → évaluation → analyse des erreurs.**

### Objectifs

À la fin du TP, vous devrez être capable de :
- traduire une question métier en problème de Machine Learning ;
- identifier la granularité d'un jeu de données ;
- repérer une fuite de données (*data leakage*) ;
- construire une baseline ;
- entraîner une première régression linéaire ;
- interpréter MAE, RMSE et \(R^2\) ;
- analyser les limites d'un modèle au-delà d'un simple score.

## 1. Avant de coder : quel problème cherchons-nous à résoudre ?

Répondez d'abord aux questions suivantes.

**Q1.** S'agit-il d'un problème supervisé ou non supervisé ? Justifiez.

**Q2.** S'agit-il d'une classification ou d'une régression ? Justifiez.

**Q3.** Quelle doit être l'unité statistique de notre jeu d'apprentissage ?

**Q4.** Quelle est la variable que nous souhaitons prédire ?

> Ne poursuivez pas immédiatement : formulez le problème sous la forme $X \rightarrow Y$.

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._




## 2. Du système d'information à la table d'apprentissage

Les données Olist proviennent d'un système transactionnel : les informations relatives à une commande sont réparties dans plusieurs tables.

Dans un SI, la granularité des tables n'est pas nécessairement celle dont un modèle de Machine Learning a besoin.

Par exemple :
- `orders` décrit les commandes ;
- `order_items` décrit les lignes de commande ;
- `products` décrit les produits ;
- `customers` décrit les clients associés aux commandes.

Une commande contenant plusieurs articles apparaît donc plusieurs fois dans `order_items`.

### Question

Pourquoi ne peut-on pas utiliser directement `order_items` comme jeu d'apprentissage si notre objectif est de prédire **un délai par commande** ?

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

### Une agrégation typique

Pour passer d'une table au niveau « article » à une table au niveau « commande », on peut agréger les lignes partageant le même `order_id`.

```python
order_items.groupby("order_id").agg(
    n_items=("order_item_id", "count"),
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum")
)
```

Cette opération illustre une étape essentielle :

$$
\boxed{\text{SI transactionnel} \neq \text{table analytique d'apprentissage}}
$$

Dans la suite, nous utiliserons une table analytique préparée à partir des différentes tables Olist.

## 3. Chargement de la table analytique `orders_ml`

La table est hébergée dans le dépôt GitHub du cours. Elle peut donc être chargée directement dans Google Colab.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "ndiayemairame/miage-machine-learning/"
    "main/data/orders_ml.csv"
)

date_columns = [
    "order_purchase_timestamp",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

orders_ml = pd.read_csv(DATA_URL, parse_dates=date_columns)

print("Dimensions :", orders_ml.shape)
orders_ml.head()

### Notre table analytique

**Granularité : une ligne représente une commande livrée.**

La table contient **96 470 commandes et 33 variables**. Certaines variables proviennent directement du SI, d'autres ont été construites par agrégation ou transformation.

Avant de modéliser, il faut comprendre ce que signifie chaque colonne.

| Variable | Description |
|---|---|
| `order_id` | Identifiant unique de la commande. |
| `customer_id` | Identifiant du client associé à cette commande dans Olist. |
| `order_purchase_timestamp` | Date et heure auxquelles la commande a été passée. |
| `order_delivered_carrier_date` | Date et heure auxquelles la commande a été remise au transporteur. |
| `order_delivered_customer_date` | Date et heure auxquelles la commande a effectivement été livrée au client. |
| `order_estimated_delivery_date` | Date de livraison estimée annoncée pour la commande. |
| `purchase_year` | Année de la commande. |
| `purchase_month` | Mois de la commande, de 1 à 12. |
| `purchase_day_of_week` | Jour de la semaine : 0 = lundi, ..., 6 = dimanche. |
| `purchase_hour` | Heure à laquelle la commande a été passée, de 0 à 23. |
| `promised_delivery_days` | Nombre de jours entre la commande et la date de livraison estimée. |
| `carrier_time_days` | Nombre de jours entre la commande et sa remise effective au transporteur. |
| `delivery_time_days` | Nombre de jours entre la commande et sa livraison effective. **Cible de ce TP.** |
| `is_late` | 1 si la livraison a eu lieu après le jour estimé, 0 sinon. |
| `customer_state` | État brésilien dans lequel se trouve le client. |
| `n_items` | Nombre de lignes/articles dans la commande. |
| `n_products` | Nombre de produits distincts. |
| `n_sellers` | Nombre de vendeurs distincts impliqués. |
| `total_price` | Prix total des articles hors transport, en réais brésiliens (BRL). |
| `total_freight` | Total des frais de transport, en réais brésiliens (BRL). |
| `mean_product_weight_g` | Poids moyen des articles, en grammes. |
| `max_product_weight_g` | Poids maximal des articles, en grammes. |
| `mean_product_volume_cm3` | Volume moyen des articles, en cm³. |
| `max_product_volume_cm3` | Volume maximal des articles, en cm³. |
| `n_categories` | Nombre de catégories de produits distinctes. |
| `main_product_category` | Catégorie la plus représentée dans la commande. |
| `freight_ratio` | Rapport `total_freight / total_price`. |
| `seller_customer_distance_mean` | Distance moyenne entre le client et les vendeurs, en km. |
| `seller_customer_distance_max` | Distance maximale entre le client et les vendeurs, en km. |
| `same_state_share` | Proportion de vendeurs situés dans le même État que le client. |
| `payment_type_main` | Mode de paiement représentant le montant payé le plus important. |
| `payment_installments_max` | Nombre maximal de mensualités associé aux paiements. |
| `review_score` | Note donnée par le client après la commande, généralement de 1 à 5. |

Vérifions la granularité et le schéma

In [ ]:
print("Nombre de lignes :", len(orders_ml))
print("Nombre de colonnes :", orders_ml.shape[1])
print("order_id unique :", orders_ml["order_id"].is_unique)

orders_ml.info()

## 4. Le temps de la prédiction : avons-nous le droit d'utiliser cette information ?

Notre modèle doit produire sa prédiction **juste après validation de la commande, une fois la date estimée de livraison communiquée au client**.

C'est ce moment précis qui détermine les informations disponibles.

Pour chaque variable ci-dessous, classez-la dans l'une des catégories :

- **A** — disponible au moment de la prédiction ;
- **B** — information provenant du futur ;
- **C** — cible ou information directement dérivée du résultat que l'on cherche à prédire.

Justifiez les cas qui vous semblent ambigus.

In [ ]:
variables_to_analyse = [
    "purchase_hour",
    "customer_state",
    "n_items",
    "total_price",
    "total_freight",
    "promised_delivery_days",
    "seller_customer_distance_mean",
    "payment_type_main",
    "order_delivered_carrier_date",
    "carrier_time_days",
    "order_delivered_customer_date",
    "review_score",
    "delivery_time_days",
]

pd.DataFrame({
    "variable": variables_to_analyse,
    "catégorie (A/B/C)": "",
    "justification": ""
})

### Proposition d'un collègue

Un collègue vous dit :

> « `carrier_time_days` est très corrélée au délai de livraison. Je propose de l'utiliser dans notre modèle. »

**Q5.** Que pensez-vous de cette proposition ? L'utiliseriez-vous ? Justifiez votre décision à partir du moment auquel notre prédiction doit être produite.

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._




## 5. Comprendre la variable cible

La variable cible $Y$ correspond à `delivery_time_days`.

Avant de construire un modèle, explorons sa distribution.

In [ ]:
target = "delivery_time_days"

orders_ml[target].describe()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(orders_ml[target], bins=60)
ax.set_xlabel("Délai réel de livraison (jours)")
ax.set_ylabel("Nombre de commandes")
ax.set_title("Distribution du délai de livraison")
plt.show()

**Q6.** Que représentent concrètement la moyenne, la médiane et les quantiles de cette variable ?

**Q7.** Observez la distribution. Est-elle symétrique ? Quelles conséquences cela peut-il avoir sur les mesures d'erreur ?

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._




## 6. Séparer apprentissage, validation et test

Nous voulons mesurer la capacité du modèle à généraliser à des commandes qu'il n'a pas utilisées pour apprendre.

Nous allons conserver :
- 70 % des observations pour l'apprentissage ;
- 15 % pour la validation et les choix expérimentaux ;
- 15 % pour le test final.

Le jeu de test restera de côté jusqu'à la fin.

In [ ]:
from sklearn.model_selection import train_test_split

train, temp = train_test_split(
    orders_ml,
    test_size=0.30,
    random_state=42
)

validation, test = train_test_split(
    temp,
    test_size=0.50,
    random_state=42
)

print(f"Train      : {len(train):,}")
print(f"Validation : {len(validation):,}")
print(f"Test       : {len(test):,}")

### Question

Pourquoi serait-il problématique de regarder continuellement les performances sur le jeu de test pendant que nous choisissons nos variables et nos modèles ?

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

## 7. Avant le Machine Learning : construire une baseline

Un modèle n'est intéressant que s'il fait mieux qu'une stratégie simple.

Imaginons un système naïf qui prédit **le même délai pour toutes les commandes**.

Nous utiliserons comme prédiction la moyenne du délai observé sur le jeu d'apprentissage.

**Q8.** Pourquoi cette moyenne doit-elle être calculée uniquement sur `train` ?

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._




In [ ]:
baseline_value = train[target].mean()
y_val = validation[target]

y_pred_baseline = np.full(len(validation), baseline_value)

print(f"Prédiction constante : {baseline_value:.2f} jours")

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }

baseline_metrics = regression_metrics(y_val, y_pred_baseline)
pd.Series(baseline_metrics)

### Interprétation

**Q9.** Traduisez la MAE en une phrase compréhensible par un responsable métier.

**Q10.** Pourquoi le \(R^2\) de cette baseline est-il proche de 0 ?

**Q11.** Pourquoi la RMSE est-elle généralement supérieure à la MAE ?

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._




## 8. Notre première régression linéaire

Commençons volontairement avec peu de variables numériques disponibles au moment de la prédiction.

**Variables du premier modèle :** `total_price`, `total_freight`, `n_items`, `promised_delivery_days`.

Le but est d'obtenir un premier modèle simple que nous pourrons comprendre et comparer à la baseline.

In [ ]:
from sklearn.linear_model import LinearRegression

features_simple = [
    "total_price",
    "total_freight",
    "n_items",
    "promised_delivery_days",
]

X_train = train[features_simple]
y_train = train[target]

X_val = validation[features_simple]

X_train.isna().sum()

### Question

Certaines variables contiennent-elles des valeurs manquantes ? Peut-on entraîner directement notre régression linéaire sur ces données ?

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._

In [ ]:
simple_model = LinearRegression()

simple_model.fit(X_train, y_train)

y_pred_simple = simple_model.predict(X_val)

simple_metrics = regression_metrics(y_val, y_pred_simple)

pd.DataFrame(
    [baseline_metrics, simple_metrics],
    index=["Baseline", "Régression linéaire"]
)

### Comprendre avant d'améliorer

**Q12.** Le modèle fait-il mieux que la baseline ? Sur quelles métriques ?

**Q13.** Que signifie ici un \(R^2\) positif mais loin de 1 ?

**Q14.** Est-ce qu'une performance imparfaite signifie nécessairement que le modèle est inutile ?

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._




## 9. Que nous dit la régression linéaire ?

Dans une régression linéaire :

$$
\hat y = \theta_0 + \theta_1 x_1 + \cdots + \theta_p x_p
$$

Observons les coefficients appris.

In [ ]:
coefficients = pd.Series(
    simple_model.coef_,
    index=features_simple,
    name="coefficient"
).sort_values()

print("Intercept :", simple_model.intercept_)
coefficients

### Questions

**Q15.** Comment interprétez-vous le signe d'un coefficient ?

**Q16.** Peut-on conclure que la variable ayant le coefficient le plus élevé est nécessairement la variable « la plus importante » ?

Pensez notamment aux unités : prix en BRL, nombre d'articles, nombre de jours...

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._




## 10. Enrichir le modèle

Notre premier modèle ignore beaucoup d'informations disponibles au moment de la commande.

Ajoutons maintenant des variables :
- calendaires ;
- liées aux produits ;
- liées à la géographie ;
- liées au paiement ;
- catégorielles comme l'État du client ou la catégorie principale.

Avant de l'entraîner, observons les nouvelles difficultés que cela introduit.


In [ ]:
numeric_features = [
    "purchase_month",
    "purchase_day_of_week",
    "purchase_hour",
    "promised_delivery_days",
    "n_items",
    "n_products",
    "n_sellers",
    "total_price",
    "total_freight",
    "mean_product_weight_g",
    "max_product_weight_g",
    "mean_product_volume_cm3",
    "max_product_volume_cm3",
    "n_categories",
    "freight_ratio",
    "seller_customer_distance_mean",
    "seller_customer_distance_max",
    "same_state_share",
    "payment_installments_max",
]

categorical_features = [
    "customer_state",
    "main_product_category",
    "payment_type_main",
]

features_rich = numeric_features + categorical_features

X_train_rich = train[features_rich]
X_val_rich = validation[features_rich]

### Avant de construire le modèle

Nous avons ajouté de nombreuses variables par rapport à notre première régression.

Commençons par vérifier si elles peuvent toutes être utilisées directement.

In [ ]:
X_train_rich.isna().sum().sort_values(ascending=False)

**Q17.** Certaines variables contiennent-elles des valeurs manquantes ? Lesquelles ?

**Q18.** Pourquoi ces valeurs manquantes peuvent-elles poser problème lors de l'entraînement du modèle ?

**Q19.** Nous avons également introduit des variables catégorielles comme `customer_state`, `main_product_category` et `payment_type_main`. Une régression linéaire peut-elle les utiliser directement sous cette forme ?

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._



### Préparer les données

Notre premier modèle utilisait uniquement des variables numériques directement exploitables.

Cette fois, deux problèmes apparaissent :

- certaines valeurs sont manquantes ;
- certaines variables sont catégorielles.

Nous devons donc préparer les données avant de pouvoir entraîner la régression.

Pour cela, nous allons utiliser trois nouveaux outils de `scikit-learn` :

- `SimpleImputer` pour traiter les valeurs manquantes ;
- `OneHotEncoder` pour transformer les variables catégorielles ;
- `Pipeline` pour enchaîner plusieurs transformations.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

### Prétraitement des variables numériques

Pour les variables numériques, nous allons remplacer les valeurs manquantes par la médiane calculée sur le jeu d'entraînement.

Le choix de la médiane permet notamment d'être moins sensible aux valeurs extrêmes que la moyenne.

In [ ]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

### Prétraitement des variables catégorielles

Pour les variables catégorielles, deux opérations sont nécessaires.

Nous remplaçons d'abord une éventuelle valeur manquante par la modalité la plus fréquente.

Puis `OneHotEncoder` transforme chaque variable catégorielle en variables numériques pouvant être utilisées par la régression.

L'option `handle_unknown="ignore"` permet également de gérer une modalité présente dans le jeu de validation mais absente du jeu d'entraînement.

In [ ]:
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

### À vous de construire le modèle

Nous disposons maintenant des deux briques nécessaires :

- `numeric_transformer` pour les variables numériques ;
- `categorical_transformer` pour les variables catégorielles.

Il reste à les assembler.

Un `ColumnTransformer` permet d'appliquer des transformations différentes à différents groupes de colonnes.

Construisez ensuite un `Pipeline` qui réalise successivement :

1. le prétraitement des données ;
2. la régression linéaire.

Entraînez enfin ce modèle sur le jeu d'entraînement et évaluez-le sur le jeu de validation.

In [ ]:
# À VOUS DE JOUER

# 1. Construisez le ColumnTransformer.
#
# Il doit appliquer :
# - numeric_transformer à numeric_features ;
# - categorical_transformer à categorical_features.

# preprocessor = ...


# 2. Construisez un Pipeline contenant :
# - le prétraitement ;
# - une régression linéaire.

# rich_model = ...


# 3. Entraînez le modèle.

# rich_model.fit(...)


# 4. Effectuez les prédictions sur le jeu de validation.

# y_pred_rich = ...


# 5. Calculez les métriques.

# rich_metrics = ...


# 6. Comparez les trois approches (baseline, modèle simple et modèle enrichi).

# pd.DataFrame(...)

### Questions

**Q20.** L'ajout de variables améliore-t-il la généralisation ?

**Q21.** Le gain est-il aussi important que vous l'imaginiez ?

**Q22.** Pourquoi « ajouter des variables » ne garantit-il pas qu'un modèle devienne meilleur ?

Nous reviendrons précisément sur cette question dans le TP suivant.

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._




## 11. Tester la proposition du collègue

Revenons à la proposition formulée précédemment : ajouter `carrier_time_days` au modèle.

Nous allons réaliser l'expérience et comparer les performances obtenues.

**Avant d'exécuter l'expérience, formulez une hypothèse : que pensez-vous qu'il va se passer ?**

In [ ]:
# À VOUS DE JOUER
#
# Reprenez la démarche utilisée pour construire le modèle enrichi,
# mais ajoutez cette fois la variable "carrier_time_days".
#
# Entraînez le nouveau modèle puis comparez ses performances
# à celles du modèle enrichi.

# 1. Construisez le nouveau ColumnTransformer.
# Utilisez les mêmes transformations que précédemment.

# leaky_preprocessor = ...


# 2. Construisez le Pipeline :
# prétraitement puis régression linéaire.

# leaky_model = ...


# 3. Entraînez le modèle.

# leaky_model.fit(...)


# 4. Effectuez les prédictions sur le jeu de validation.

# y_pred_leaky = ...


# 5. Calculez les métriques.

# leaky_metrics = ...


# 6. Comparez le modèle enrichi et ce nouveau modèle.

# pd.DataFrame(...)

### Analyse de l'expérience

**Q23.** Comparez les performances des deux modèles. Que constatez-vous ?

**Q24.** À partir de la définition de `carrier_time_days` et du moment où la prédiction doit être produite, ce modèle pourrait-il réellement être utilisé dans notre scénario métier ? Justifiez.

**Q25.** Que vous apprend cette expérience sur la relation entre performance mesurée et validité d'un protocole de Machine Learning ?

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._




## 12. Regarder les erreurs, pas seulement le score

Les métriques donnent une vision globale des performances du modèle, mais elles ne nous disent pas **sur quelles commandes le modèle se trompe**.

Nous allons donc examiner les erreurs individuellement.

Pour chaque commande, nous calculons :

- la valeur réelle du délai de livraison ;
- la valeur prédite par le modèle ;
- l'erreur de prédiction ;
- la valeur absolue de cette erreur.

Nous commencerons par observer les commandes pour lesquelles l'erreur est la plus importante.

In [ ]:
results = validation[
    ["order_id", "delivery_time_days"]
].copy()

results["prediction"] = y_pred_rich

results["error"] = (
    results["delivery_time_days"] - results["prediction"]
)

results["absolute_error"] = results["error"].abs()

results.sort_values(
    "absolute_error",
    ascending=False
).head(10)

### Analyse

**Q26.** Observez les commandes ayant les erreurs absolues les plus importantes. Que remarquez-vous ?

**Q27.** Que signifie une erreur positive ? Et une erreur négative ?

**Q28.** Les erreurs les plus importantes correspondent-elles plutôt à des délais de livraison courts ou longs ?

**Q29.** À votre avis, pourquoi ces commandes sont-elles particulièrement difficiles à prédire ?

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._



In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(
    results["delivery_time_days"],
    results["prediction"],
    alpha=0.15,
    s=10
)

limit = max(
    results["delivery_time_days"].quantile(0.99),
    results["prediction"].quantile(0.99)
)

ax.plot([0, limit], [0, limit], linestyle="--")
ax.set_xlim(0, limit)
ax.set_ylim(0, limit)
ax.set_xlabel("Délai réel (jours)")
ax.set_ylabel("Délai prédit (jours)")
ax.set_title("Valeurs réelles vs prédictions")
plt.show()

### Questions

**Q30.** Que représenterait un point situé exactement sur la diagonale ?

**Q31.** Dans quelles zones le modèle semble-t-il le plus en difficulté ?

**Q32.** Les commandes exceptionnellement longues semblent-elles bien prédites ?

**Q33.** Que pourrait-on examiner dans les données pour comprendre ces erreurs ?

### ✍️ Votre réponse

_Double-cliquez ici puis rédigez votre réponse._




## 13. Évaluation finale sur le jeu de test

Nous avons utilisé le jeu de validation pour comparer nos choix. Une fois le modèle légitime retenu, nous pouvons effectuer **une évaluation finale** sur le jeu de test laissé de côté.

Pour exploiter toutes les données disponibles avant le test, nous réentraînons le pipeline sur `train + validation`.

In [ ]:
# À VOUS DE JOUER
# 1. Regroupez train et validation.
# 2. Réentraînez le modèle enrichi.
# 3. Prédisez sur test.
# 4. Calculez les métriques finales.
#
# Le jeu de test n'a pas servi aux choix précédents.

# train_validation = ...

# final_model = ...

# final_model.fit(...)

# y_test = ...
# y_pred_test = ...

# test_metrics = ...

## 14. Bilan

Nous sommes partis d'un système d'information relationnel pour arriver à une première prédiction.

Le chemin parcouru est plus important que l'appel à `fit()` :

$$
\boxed{
\text{Question métier}
\rightarrow
\text{unité statistique}
\rightarrow
X,Y
\rightarrow
\text{temps de prédiction}
\rightarrow
\text{split}
\rightarrow
\text{baseline}
\rightarrow
\text{modèle}
\rightarrow
\text{évaluation}
}
$$

### À retenir

Un modèle doit être évalué sur des données qu'il n'a pas utilisées pour apprendre. Une variable très prédictive peut être inutilisable si elle provient du futur. Enfin, une métrique n'a de sens que si l'on sait l'interpréter dans le contexte du problème.

### Pour préparer le TP2

Notre régression reste limitée. Cela soulève plusieurs questions :

- les relations sont-elles réellement linéaires ?
- ajouter des variables améliore-t-il toujours la généralisation ?
- certaines variables devraient-elles être transformées ?
- comment contrôler la complexité d'un modèle ?
- comment reconnaître le surapprentissage ?